# Проверка центральной предельной теоремы

В этом задании мы проверим работу центральной предельной теоремы (ЦПТ) на примере распределения, отличного от нормального. Мы будем использовать **распределение Парето** (Pareto distribution), которое имеет тяжелый правый хвост и сильно асимметрично.

## Параметры распределения Парето

Распределение Парето задается двумя параметрами:
- $x_m$ (scale) — минимальное возможное значение (выберем $x_m = 1$)
- $\alpha$ (shape) — параметр формы (выберем $\alpha = 3$, чтобы дисперсия была конечной)

Теоретические характеристики:
- Математическое ожидание: $E[X] = \frac{\alpha x_m}{\alpha - 1}$ при $\alpha > 1$
- Дисперсия: $Var(X) = \frac{\alpha x_m^2}{(\alpha - 1)^2 (\alpha - 2)}$ при $\alpha > 2$

Для наших параметров ($x_m = 1, \alpha = 3$):
- $E[X] = \frac{3 \cdot 1}{3 - 1} = 1.5$
- $Var(X) = \frac{3 \cdot 1^2}{(3 - 1)^2 (3 - 2)} = \frac{3}{4} = 0.75$
- Стандартное отклонение: $\sigma = \sqrt{0.75} \approx 0.866$

In [ ]:
# Импорт необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Установка стиля графиков
plt.style.use('seaborn-v0_8-whitegrid')

# Параметры распределения Парето
xm = 1.0  # scale parameter
alpha = 3.0  # shape parameter

# Теоретические значения
theoretical_mean = (alpha * xm) / (alpha - 1)
theoretical_var = (alpha * xm**2) / ((alpha - 1)**2 * (alpha - 2))
theoretical_std = np.sqrt(theoretical_var)

print(f"Теоретическое математическое ожидание: {theoretical_mean:.4f}")
print(f"Теоретическая дисперсия: {theoretical_var:.4f}")
print(f"Теоретическое стандартное отклонение: {theoretical_std:.4f}")

## Генерация исходной выборки и построение гистограммы

Сгенерируем выборку объема 1000 из распределения Парето и построим гистограмму с наложенной теоретической плотностью.

In [ ]:
# Генерация выборки объема 1000
np.random.seed(42)  # для воспроизводимости
sample_size = 1000
sample = (np.random.pareto(alpha, sample_size) + 1) * xm

# Построение гистограммы и теоретической плотности
fig, ax = plt.subplots(figsize=(10, 6))

# Гистограмма с normed=True (density в новых версиях)
ax.hist(sample, bins=30, density=True, alpha=0.7, color='skyblue', 
        edgecolor='black', label='Гистограмма выборки')

# Теоретическая плотность распределения Парето
x_vals = np.linspace(xm, sample.max(), 1000)
pdf_vals = (alpha * xm**alpha) / (x_vals**(alpha + 1))
ax.plot(x_vals, pdf_vals, 'r-', linewidth=2, label='Теоретическая плотность')

ax.set_xlabel('Значение')
ax.set_ylabel('Плотность вероятности')
ax.set_title('Гистограмма выборки из распределения Парето и теоретическая плотность')
ax.legend()
plt.tight_layout()
plt.show()

## Проверка центральной предельной теоремы

Согласно ЦПТ, выборочное среднее $\bar{X}_n$ при больших $n$ приближается к нормальному распределению:

$$\bar{X}_n \sim N\left(\mu, \frac{\sigma^2}{n}\right)$$

где:
- $\mu$ — теоретическое математическое ожидание исходного распределения
- $\sigma^2$ — теоретическая дисперсия исходного распределения
- $n$ — объем выборки

Параметры аппроксимирующего нормального распределения:
- Среднее: $\mu_{\bar{X}} = \mu$
- Стандартное отклонение: $\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{n}}$

Рассмотрим три значения n: 5, 10, 50

In [ ]:
# Параметры для эксперимента
n_values = [5, 10, 50]  # размеры выборок
num_simulations = 1000  # количество симуляций для каждого n

# Создание фигуры с подграфиками
fig, axes = plt.subplots(1, len(n_values), figsize=(15, 5))

for i, n in enumerate(n_values):
    # Генерация num_simulations выборок объема n и вычисление их средних
    sample_means = []
    for _ in range(num_simulations):
        sample = (np.random.pareto(alpha, n) + 1) * xm
        sample_means.append(np.mean(sample))
    
    sample_means = np.array(sample_means)
    
    # Параметры аппроксимирующего нормального распределения по ЦПТ
    mu_normal = theoretical_mean
    sigma_normal = theoretical_std / np.sqrt(n)
    
    print(f"\nn = {n}:")
    print(f"  Теоретическое среднее выборочного среднего: {mu_normal:.4f}")
    print(f"  Теоретическое стандартное отклонение выборочного среднего: {sigma_normal:.4f}")
    print(f"  Выборочное среднее экспериментальных средних: {np.mean(sample_means):.4f}")
    print(f"  Выборочное стандартное отклонение экспериментальных средних: {np.std(sample_means):.4f}")
    
    # Построение гистограммы
    ax = axes[i]
    ax.hist(sample_means, bins=30, density=True, alpha=0.7, color='lightgreen',
            edgecolor='black', label=f'Гистограмма (n={n})')
    
    # Плотность нормального распределения по ЦПТ
    x_vals = np.linspace(sample_means.min(), sample_means.max(), 1000)
    normal_pdf = stats.norm.pdf(x_vals, loc=mu_normal, scale=sigma_normal)
    ax.plot(x_vals, normal_pdf, 'r-', linewidth=2, 
            label=f'Нормальное распределение\nN({mu_normal:.2f}, {sigma_normal:.2f})')
    
    ax.set_xlabel('Выборочное среднее')
    ax.set_ylabel('Плотность вероятности')
    ax.set_title(f'Распределение выборочных средних (n={n})')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Вычисления параметров нормальных распределений

Для каждого значения n вычислим параметры нормального распределения, которым по ЦПТ приближается распределение выборочных средних:

**Исходные данные:**
- $\mu = 1.5$
- $\sigma = \sqrt{0.75} \approx 0.8660$

**Для n = 5:**
- Среднее: $\mu_{\bar{X}} = \mu = 1.5$
- Стандартное отклонение: $\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{5}} = \frac{0.8660}{\sqrt{5}} \approx 0.3873$

**Для n = 10:**
- Среднее: $\mu_{\bar{X}} = \mu = 1.5$
- Стандартное отклонение: $\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{10}} = \frac{0.8660}{\sqrt{10}} \approx 0.2739$

**Для n = 50:**
- Среднее: $\mu_{\bar{X}} = \mu = 1.5$
- Стандартное отклонение: $\sigma_{\bar{X}} = \frac{\sigma}{\sqrt{50}} = \frac{0.8660}{\sqrt{50}} \approx 0.1225$

In [ ]:
# Точные вычисления параметров
print("Параметры аппроксимирующих нормальных распределений:")
print("="*60)

for n in n_values:
    mu_bar = theoretical_mean
    sigma_bar = theoretical_std / np.sqrt(n)
    print(f"n = {n:2d}: μ = {mu_bar:.6f}, σ = {sigma_bar:.6f}")

## Анализ результатов и выводы

### Наблюдаемые закономерности:

1. **Форма распределения:**
   - При n = 5: распределение выборочных средних всё ещё заметно асимметрично, хотя уже начинает напоминать нормальное
   - При n = 10: асимметрия уменьшается, форма становится более симметричной
   - При n = 50: распределение практически неотличимо от нормального

2. **Разброс значений:**
   - С ростом n стандартное отклонение выборочных средних уменьшается пропорционально $1/\sqrt{n}$
   - Это означает, что оценки становятся более точными и концентрируются вокруг истинного значения μ

3. **Точность аппроксимации:**
   - Чем больше n, тем лучше нормальное распределение аппроксимирует распределение выборочных средних
   - Уже при n = 50 аппроксимация очень хорошая, несмотря на сильную асимметрию исходного распределения Парето

### Выводы:

Центральная предельная теорема подтверждается экспериментально:
- Независимо от формы исходного распределения (в данном случае сильно асимметричного распределения Парето), распределение выборочных средних стремится к нормальному с ростом объема выборки n
- Параметры предельного нормального распределения точно соответствуют теоретическим предсказаниям: среднее равно μ, стандартное отклонение равно σ/√n
- Для практических целей уже при n ≈ 30-50 аппроксимация нормальным распределением работает очень хорошо даже для сильно не-нормальных исходных распределений
- Это объясняет, почему нормальное распределение так широко встречается в природе и практике: многие наблюдаемые величины являются средними большого числа независимых случайных факторов